U-DET

Loading the drive

In [1]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


U Net Definition




In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        # 1. Codificador (Encoder / Downsampling)
        self.enc1 = self.conv_block(1, 64)   # Entrada: CEM (1 canal)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 2. Cuello de botella (Bottleneck)
        self.bottleneck = self.conv_block(256, 512)

        # 3. Decodificador (Decoder / Upsampling)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = self.conv_block(512, 256) # 512 porque concatena con Skip Connection

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(128, 64)

        # 4. Capa de Salida
        self.final_conv = nn.Conv2d(64, 1, kernel_size=1) # Salida: Mapa de Riesgo

    def conv_block(self, in_ch, out_ch):
        """Bloque de doble convolución con normalización de batch"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Paso por el codificador y guardado de conexiones para Skip Connections
        s1 = self.enc1(x)
        p1 = self.pool(s1)

        s2 = self.enc2(p1)
        p2 = self.pool(s2)

        s3 = self.enc3(p2)
        p3 = self.pool(s3)

        b = self.bottleneck(p3)

        # Paso por el decodificador con concatenación
        d3 = self.up3(b)
        # Ajuste de tamaño por si hay diferencias de píxeles
        d3 = torch.cat((d3, s3), dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat((d2, s2), dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat((d1, s1), dim=1)
        d1 = self.dec1(d1)

        return torch.sigmoid(self.final_conv(d1)) # Sigmoide para probabilidad de riesgo

# Inicializar modelo
model = UNet()
print("Arquitectura U-Net inicializada correctamente.")

Arquitectura U-Net inicializada correctamente.
